Catastrophic cancellation. Compute the variance of [1000000.0, 1000001.0, 1000002.0] using the naive formula E[x^2] - E[x]^2 in float32. Then compute it using Welford's online algorithm. Compare the errors against the true variance (0.6667).

In [1]:
import numpy as np
data = [1000000.0, 1000001.0, 1000002.0]
n=len(data)
true_variance=np.var(np.array(data,dtype=np.float64))
print(f"True variance (float64): {true_variance:.10f}")
print(f"Expected: 0.6667")
print()

True variance (float64): 0.6666666667
Expected: 0.6667



- Compute E[x] and E[x^2]


In [2]:
data_f32 = np.array(data, dtype=np.float32)
mean_f32 = np.mean(data_f32)
mean_sq_f32 = np.mean(data_f32**2)
naive_variance = mean_sq_f32 - mean_f32**2
print("=== Naive Formula (float32) ===")
print(f"E[x]   = {mean_f32}")
print(f"E[x^2] = {mean_sq_f32}")
print(f"E[x]^2 = {mean_f32**2}")
print(f"Variance = E[x^2] - E[x]^2 = {naive_variance}")
print(f"Error: {abs(naive_variance - true_variance):.6f}")
print(f"Relative error: {abs(naive_variance - true_variance)/true_variance * 100:.2f}%")
print()

=== Naive Formula (float32) ===
E[x]   = 1000001.0
E[x^2] = 1000001961984.0
E[x]^2 = 1000002027520.0
Variance = E[x^2] - E[x]^2 = -65536.0
Error: 65536.666667
Relative error: 9830500.00%



- Method 2: Welford's online algorithm in float32

In [3]:
def welford_variance(data):
    n = 0
    mean = np.float32(0.0)
    M2 = np.float32(0.0)
    
    for x in data:
        x = np.float32(x)
        n += 1
        delta = x - mean
        mean += delta / np.float32(n)
        delta2 = x - mean
        M2 += delta * delta2
    
    if n < 2:
        return np.float32(0.0)
    else:
        return M2 / np.float32(n)

In [4]:
welford_var = welford_variance(data)

print("=== Welford's Online Algorithm (float32) ===")
print(f"Variance = {welford_var}")
print(f"Error: {abs(welford_var - true_variance):.6f}")
print(f"Relative error: {abs(welford_var - true_variance)/true_variance * 100:.6f}%")
print()

=== Welford's Online Algorithm (float32) ===
Variance = 0.6666666865348816
Error: 0.000000
Relative error: 0.000003%



In [5]:
# Compare catastrophic cancellation
print("=== Analysis ===")
print(f"Naive formula suffers from catastrophic cancellation:")
print(f"  E[x^2] = {mean_sq_f32} (~1e12)")
print(f"  E[x]^2 = {mean_f32**2} (~1e12)")
print(f"  Difference = {naive_variance} (should be ~0.667)")
print(f"  We're subtracting two large nearly-equal numbers,")
print(f"  losing ~6-7 significant digits in float32 (which has ~7 digits).")
print()
print(f"Welford's algorithm avoids this by working with deviations")
print(f"from the running mean, keeping numbers small and well-conditioned.")

=== Analysis ===
Naive formula suffers from catastrophic cancellation:
  E[x^2] = 1000001961984.0 (~1e12)
  E[x]^2 = 1000002027520.0 (~1e12)
  Difference = -65536.0 (should be ~0.667)
  We're subtracting two large nearly-equal numbers,
  losing ~6-7 significant digits in float32 (which has ~7 digits).

Welford's algorithm avoids this by working with deviations
from the running mean, keeping numbers small and well-conditioned.


Precision hunt. Find the smallest positive float32 value x such that 1.0 + x == 1.0 in Python. This is the machine epsilon. Verify it matches numpy.finfo(numpy.float32).eps

In [6]:
import numpy as np

# Method 1: Binary search / iterative approach to find smallest x where 1.0 + x == 1.0
def find_epsilon_iterative():
    """Find the smallest positive float32 x such that 1.0 + x == 1.0"""
    x = np.float32(1.0)
    
    # Keep halving until 1.0 + x == 1.0
    while np.float32(1.0) + x != np.float32(1.0):
        x = np.float32(x / 2)
    
    # Now x is the largest value where 1.0 + x != 1.0
    # The machine epsilon is the next representable value up (2*x)
    # Actually, let's be more careful:
    # We want the SMALLEST x such that 1.0 + x == 1.0
    # That's exactly the value we found (the first one that rounds to 1.0)
    return x

# Better approach: start from 1.0 and halve
def find_epsilon():
    """Find smallest positive float32 x such that 1.0 + x == 1.0"""
    x = np.float32(1.0)
    prev = x
    
    # Halve until 1.0 + x == 1.0
    while np.float32(1.0) + x != np.float32(1.0):
        prev = x
        x = np.float32(x / 2)
    
    # x is the smallest value where 1.0 + x == 1.0
    # prev is the largest value where 1.0 + prev != 1.0
    return x, prev

eps_found, largest_non_eps = find_epsilon()

print("=== Finding Machine Epsilon for float32 ===")
print(f"Smallest x where 1.0 + x == 1.0: {eps_found}")
print(f"Largest x where 1.0 + x != 1.0:  {largest_non_eps}")
print()

# Verify: the standard definition of machine epsilon is the difference
# between 1.0 and the next representable float
next_float = np.nextafter(np.float32(1.0), np.float32(2.0))
eps_nextafter = next_float - np.float32(1.0)

print(f"np.nextafter(1.0, 2.0) - 1.0 = {eps_nextafter}")
print()

# Compare with numpy.finfo
eps_numpy = np.finfo(np.float32).eps

print("=== Comparison ===")
print(f"Found epsilon (1.0 + x == 1.0):  {eps_found}  ({eps_found:.10e})")
print(f"nextafter-based epsilon:          {eps_nextafter}  ({eps_nextafter:.10e})")
print(f"numpy.finfo(float32).eps:         {eps_numpy}  ({eps_numpy:.10e})")
print()

print(f"Match (found == numpy):     {eps_found == eps_numpy}")
print(f"Match (nextafter == numpy): {eps_nextafter == eps_numpy}")
print()

# Let's verify the boundary conditions
print("=== Boundary Verification ===")
eps = eps_numpy
print(f"1.0 + eps         = {np.float32(1.0) + eps}  (should be != 1.0? No, should be > 1.0)")
print(f"1.0 + eps/2       = {np.float32(1.0) + np.float32(eps/2)}  (should be == 1.0)")
print(f"1.0 + eps*(1-δ)   = {np.float32(1.0) + np.float32(eps * np.float32(0.999))}  (rounds to 1.0)")
print()

# More precise check
print("=== Precise Boundary Check ===")
# Just below eps
just_below = np.nextafter(eps, np.float32(0))
print(f"eps                    = {eps:.10e}")
print(f"nextafter(eps, 0)      = {just_below:.10e}")
print(f"1.0 + eps              = {np.float32(1.0) + eps}")
print(f"1.0 + just_below       = {np.float32(1.0) + just_below}")
print(f"1.0 + just_below == 1.0? {np.float32(1.0) + just_below == np.float32(1.0)}")
print()

# Understanding: eps is the gap between 1.0 and the next float
print("=== Understanding the Result ===")
print(f"Representation of 1.0 in float32: exact")
print(f"Next float32 after 1.0: {next_float}")
print(f"Gap: {eps:.10e}")
print(f"This gap = 2^(-23) = {2.0**-23:.10e}")
print(f"Float32 has 23 mantissa bits, so eps = 2^-23 = {2.0**-23}")
print()
print(f"np.finfo(np.float32).eps = 2^-23? {eps == np.float32(2.0**-23)}")
print(f"np.finfo(np.float32).nmant = {np.finfo(np.float32).nmant}")

=== Finding Machine Epsilon for float32 ===
Smallest x where 1.0 + x == 1.0: 5.960464477539063e-08
Largest x where 1.0 + x != 1.0:  1.1920928955078125e-07

np.nextafter(1.0, 2.0) - 1.0 = 1.1920928955078125e-07

=== Comparison ===
Found epsilon (1.0 + x == 1.0):  5.960464477539063e-08  (5.9604644775e-08)
nextafter-based epsilon:          1.1920928955078125e-07  (1.1920928955e-07)
numpy.finfo(float32).eps:         1.1920928955078125e-07  (1.1920928955e-07)

Match (found == numpy):     False
Match (nextafter == numpy): True

=== Boundary Verification ===
1.0 + eps         = 1.0000001192092896  (should be != 1.0? No, should be > 1.0)
1.0 + eps/2       = 1.0  (should be == 1.0)
1.0 + eps*(1-δ)   = 1.0000001192092896  (rounds to 1.0)

=== Precise Boundary Check ===
eps                    = 1.1920928955e-07
nextafter(eps, 0)      = 1.1920928245e-07
1.0 + eps              = 1.0000001192092896
1.0 + just_below       = 1.0000001192092896
1.0 + just_below == 1.0? False

=== Understanding the Resu